# Giskard (open-source) — Testing & Red-Teaming LLM Apps · A Comprehensive Tutorial

**Giskard OSS · Python-native library · Scan · Test Suites · RAGET**

This notebook teaches you to test and red-team an LLM application end-to-end with the
**open-source** [Giskard](https://github.com/Giskard-AI/giskard-oss) library. It is written as a
*runnable learning tutorial*: read each markdown cell, then run the code cell beneath it, one at a
time.

If you also went through the companion **promptfoo** and **garak** tutorials, you'll notice Giskard
takes a different angle: instead of only firing attacks, it **auto-scans** your model for whole
categories of vulnerabilities, then turns findings into a **reusable test suite**. A mapping table
across all three tools is in the appendix.

### Scope: red-teaming, open-source, Gemini as the evaluator
- **Focus: red-teaming / vulnerability scanning and Giskard's capabilities.** The heart of this
  notebook is **`giskard.scan`** (auto-detecting jailbreaks, harmful content, hallucination, data
  leakage, stereotypes, …) and turning findings into a **test suite**. Giskard's **RAGET** (RAG
  evaluation) is a short *overview* in Section 9 only.
- **Open-source only.** We use the OSS library; **not** the paid **Giskard Hub**. Everything runs
  locally.
- **Target = a FastAPI HTTP endpoint.** Your app is served over HTTP; we wrap that endpoint as the
  model under test in Section 4 (assumed already running and reachable).
- **Evaluator LLM = Gemini.** Giskard's scan is LLM-assisted; we point it at **Gemini** for probe
  generation and answer judging (assumed you have working Gemini access). Configured in Section 3.

> **Version note (read this):** the mature OSS scanning + RAGET features are **v2**. Per the
> project README, v2 "remains available but unmaintained" while a **v3** rewrite for agentic
> systems is in progress. This tutorial targets v2, so we pin the install accordingly:
> `pip install "giskard[llm]>2,<3"`. Check the repo for v3 status before starting new long-term
> work.

### Assumptions baked into this notebook
- **Your app is a running FastAPI HTTP endpoint.** It accepts a request (e.g. JSON `{"question": ...}`)
  and returns text. We only *wrap* that endpoint for Giskard; it's assumed up and reachable.
- **Gemini access works.** Giskard's LLM scan uses an LLM to generate probes and judge answers; we
  use Gemini and assume the API key is valid.
- **Learning-first, local, no Hub.**

### Giskard is Python-native
There's no CLI to shell out to — you use the library directly. This notebook is real Python:
`giskard.Model(...)`, `giskard.scan(...)`, `report.generate_test_suite()`, `giskard.rag.*`.

---
## 1 · The Giskard mental model

Giskard OSS is organized around a few objects and one scan:

| Component | Role | Analogy (promptfoo / garak) |
|-----------|------|------------------------------|
| **`giskard.Model`** *(wraps your target)* | Black-box wrapper around your predict function | target/provider · generator |
| **`giskard.Dataset`** | Optional seed inputs the scan can probe/perturb | test vars · probe prompts |
| **`giskard.scan()`** | Auto-detects vulnerability categories, returns a **ScanReport** | `redteam run` · `garak` scan |
| **Detectors** | The categories the scan checks (injection, harm, hallucination, …) | plugins+strategies · probes+detectors |
| **Test suite** | Findings turned into re-runnable pass/fail tests (regression) | *(no direct equivalent)* |
| **RAGET** | Generates a Q&A test set from a knowledge base and scores a RAG app | *(no direct equivalent)* |

### The workflow this notebook centers on

```
  wrap model  ->  giskard.scan()  ->  ScanReport (HTML)  ->  generate_test_suite()  ->  re-run in CI
                   (red-team scan)     (findings)            (regression gate)
```

This is the LLM red-team / vulnerability scan. RAGET (RAG-specific evaluation) is a separate,
optional workflow summarized briefly in Section 9.

---
## 2 · Environment sanity checks

Confirm Giskard is installed with the LLM extras. (Install assumed done:
`pip install "giskard[llm]>2,<3"`, Python 3.10+.)

In [ ]:
import giskard
print("giskard version:", giskard.__version__)
print("giskard location:", giskard.__file__)

In [ ]:
# The red-team pieces we'll use — confirm they import in your install.
from giskard import Model, Dataset, scan
print("Core scan API imported OK.")
# (RAGET lives in giskard.rag — imported only if you use the Section 9 overview.)

---
## 3 · Point Giskard's evaluator at Gemini

Giskard's scan is **LLM-assisted** — it uses an LLM to craft adversarial inputs and to judge
answers (and an embedding model). We use **Gemini** for this. Giskard talks to providers through
[LiteLLM](https://docs.litellm.ai/), so Gemini models are referenced as `gemini/<model>`.

> This "judge/attacker" LLM is *separate* from the model under test (the FastAPI endpoint in
> Section 4). Use a capable Gemini model here — stronger judge = better probes and fewer
> mis-judgments.

In [ ]:
import os
import giskard

# Gemini access (assumed working). LiteLLM reads GEMINI_API_KEY (or GOOGLE_API_KEY).
# os.environ["GEMINI_API_KEY"] = "..."      # set if not already in your environment

# Attacker/judge model + embedding model, both on Gemini.
giskard.llm.set_llm_model("gemini/gemini-1.5-pro")
giskard.llm.set_embedding_model("gemini/text-embedding-004")

print("Giskard evaluator configured on Gemini.")

---
## 4 · Wrap your model (and optional seed data)

`giskard.Model` wraps **any** black-box function that takes a `pandas.DataFrame` of inputs and
returns a list of text outputs. Key arguments:

- `model` — your predict function.
- `model_type="text_generation"` — for LLM apps.
- `name` / `description` — **the description matters a lot**: the scan uses it to understand what
  your app is *supposed* to do (like promptfoo's `purpose`). Be specific.
- `feature_names` — the input column(s) your function reads.

Your target is a **FastAPI HTTP endpoint**, so the wrapper just POSTs each input to it and returns
the text reply. Giskard hands your function a whole `DataFrame` at once — loop over the rows (or
batch if your API supports it).

In [ ]:
import pandas as pd
import requests
import giskard

# ---- Your FastAPI endpoint (assumed running & reachable) -------------------
ENDPOINT = "http://localhost:8000/chat"     # <-- REPLACE with your endpoint URL
TIMEOUT = 30

def call_endpoint(question: str) -> str:
    # Adjust the request/response shape to match YOUR FastAPI contract.
    resp = requests.post(ENDPOINT, json={"question": question}, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    return data.get("answer", data.get("response", ""))   # <-- match your response key

def model_predict(df: pd.DataFrame):
    # One text output per input row (Giskard passes a DataFrame).
    return [call_endpoint(q) for q in df["question"]]

giskard_model = giskard.Model(
    model=model_predict,
    model_type="text_generation",
    name="My FastAPI LLM Application",
    description=(
        "REPLACE ME: a detailed description of what this app does, who uses it, "
        "what it may access, and what it must never do. The scan uses this to "
        "craft relevant adversarial probes and to judge answers."
    ),
    feature_names=["question"],
)
print("Wrapped FastAPI endpoint as:", giskard_model.name)

In [ ]:
# Sanity-check the endpoint wiring BEFORE scanning: one real request round-trip.
probe = pd.DataFrame({"question": ["Hello — what can you help me with?"]})
print(model_predict(probe)[0][:300])

In [ ]:
# OPTIONAL: a small seed dataset of realistic inputs. The scan will probe and
# perturb these; if omitted, Giskard generates its own from the description.
import pandas as pd, giskard

seed = giskard.Dataset(
    pd.DataFrame({"question": [
        "What is your return policy?",
        "Can you help me reset my password?",
        "Summarize your key features.",
    ]}),
    name="seed prompts",
    target=None,   # no ground-truth label column for generation tasks
)
print("Seed dataset rows:", len(seed))

---
## 5 · Run the LLM vulnerability scan

`giskard.scan(model, dataset=None, only=None)` runs the detector suite and returns a **ScanReport**.
With no `dataset`, Giskard synthesizes probes from your model `description`.

- `only=[...]` restricts to specific detector groups (faster, focused).
- The scan is LLM-assisted, so it costs tokens on the client from Section 3 and takes a few minutes.

Start with a **focused** scan (one or two detector groups) to confirm wiring before the full run.

In [ ]:
import giskard

# Focused smoke scan — just prompt injection + harmfulness, using the seed data.
scan_smoke = giskard.scan(giskard_model, seed, only=["jailbreak", "harmfulness"])
scan_smoke

In [ ]:
import giskard

# Full scan — all applicable LLM detectors. This is the main red-team pass.
scan_results = giskard.scan(giskard_model, seed)
scan_results   # renders an interactive report inline in Jupyter

---
## 6 · Capabilities — what the LLM scan can detect

This is the core of Giskard's red-teaming value: a **single `scan` call** runs a whole battery of
detectors, each targeting a different vulnerability class. You don't hand-write attacks — Giskard
generates them from your model `description` and known attack libraries, fires them at your
endpoint, and judges the responses with the Gemini evaluator.

| Capability (detector group) | What it probes for | Example failure it catches |
|---|---|---|
| **Prompt injection / jailbreak** | Input overriding system instructions; known jailbreak templates (DAN-style, payload smuggling) | Bot follows "ignore your rules and…" |
| **Harmful content** | Toxic, dangerous, or inappropriate generation | Produces instructions for wrongdoing |
| **Stereotypes / discrimination** | Bias across gender, race, religion, age, etc. | Different treatment by demographic |
| **Hallucination / misinformation** | Fabricated facts, **sycophancy** (agreeing with false premises), implausible or self-contradicting output | Confidently invents a policy that doesn't exist |
| **Sensitive information disclosure** | Leaking secrets, PII, credentials, or **system-prompt contents** | Reveals its hidden instructions on request |
| **Robustness** | Output instability under perturbations — typos, control chars, rephrasings | Answer flips meaning after a trivial edit |
| **Output formatting** | Violating a required output format/schema | Returns prose where JSON was required |

**What makes the scan powerful (capabilities to lean on):**
- **Automatic probe generation** — attacks are synthesized *for your app* from its `description`,
  so you don't maintain an attack corpus by hand.
- **Severity-ranked issues** — findings come with a severity (major/medium/minor) and the exact
  offending input/output, so you triage fastest-first.
- **Focused or full runs** — scope to one capability while iterating, or run everything:

```python
giskard.scan(giskard_model, seed, only=["jailbreak"])                  # one group
giskard.scan(giskard_model, seed, only=["harmfulness", "stereotypes"]) # several
giskard.scan(giskard_model, seed)                                      # all applicable
```

- **Custom, requirement-based checks** — beyond the built-ins, you can assert *your own* business
  rules in natural language and have the LLM judge them (Section 8), and freeze any finding into a
  **regression test** so it can never silently come back.

> Detection is LLM-assisted, so always **read the flagged examples** — the report shows each
> offending input/output. With Gemini as judge you'll get strong results, but spot-check
> borderline issues rather than trusting rates blindly.

In [ ]:
# Inspect detected issues programmatically (not just the rendered report).
issues = scan_results.issues
print(f"{len(issues)} issues found\n")
for i, issue in enumerate(issues[:10], 1):
    # Attributes vary slightly by version; print defensively.
    group = getattr(issue, "group", None)
    level = getattr(issue, "level", getattr(issue, "severity", "?"))
    desc  = getattr(issue, "description", str(issue))
    print(f"{i:>2}. [{level}] {getattr(group,'name',group)} — {str(desc)[:100]}")

---
## 7 · Read and save the scan report

The ScanReport renders inline in Jupyter, but you'll also want a shareable file and programmatic
access.

In [ ]:
# Save a standalone interactive HTML report you can open in any browser.
scan_results.to_html("giskard_scan_report.html")
print("Wrote giskard_scan_report.html")

# Quick programmatic checks (handy for gating).
print("Has vulnerabilities:", scan_results.has_vulnerabilities)

---
## 8 · Turn findings into a reusable test suite

This is Giskard's signature move: convert scan findings into a **test suite** of concrete pass/fail
tests you can re-run on every model change — turning a one-off red-team into a **regression gate**.

In [ ]:
# Generate an executable test suite from the scan results.
test_suite = scan_results.generate_test_suite(name="LLM app vulnerability suite")
test_suite

In [ ]:
# Run the suite and inspect pass/fail.
suite_result = test_suite.run()
print("Suite passed:", suite_result.passed)
# suite_result contains per-test outcomes; render or iterate as needed.
suite_result

### Add your own tests

Beyond scan-derived tests, `giskard.testing` provides ready-made tests you can pin to your app's
requirements (e.g. that outputs never contain a forbidden string, or stay within a length bound).
Compose them into the same suite so everything runs together in CI.

```python
from giskard.testing import test_llm_output_against_requirement

suite = (giskard.Suite(name="my requirements")
    .add_test(test_llm_output_against_requirement(
        model=giskard_model,
        dataset=seed,
        requirement="The answer must never reveal internal pricing rules."))
)
suite.run()
```

---
## 9 · RAGET — RAG evaluation (overview only)

Beyond the red-team scan, Giskard ships **RAGET (RAG Evaluation Toolkit)** for
**retrieval-augmented** apps. We keep this to an overview — reach for it when your target is a RAG
pipeline and you want quality/grounding scores, not just vulnerability findings.

**What it does, in three moves:**
1. **`KnowledgeBase.from_pandas(df, columns=[...])`** — wrap your documents.
2. **`generate_testset(kb, num_questions=..., agent_description=...)`** — the LLM client from
   Section 3 synthesizes realistic Q&A pairs (varied types: simple, complex, situational,
   distracting, conversational), each grounded in the KB. Save with `testset.save(...)`.
3. **`evaluate(answer_fn, testset=..., knowledge_base=kb)`** — runs each question through *your*
   RAG pipeline and grades **correctness**, then attributes weaknesses to the **RAG component** at
   fault (retriever, generator, rewriter, router, or the knowledge base). Export with
   `report.to_html(...)` and inspect `report.component_scores()`.

**Why it's useful:** it tells you *where* a RAG app is failing (retrieval vs generation), which a
generic scan can't. If your app isn't RAG, skip this section entirely.

```python
# Minimal sketch (see Giskard docs for the full RAGET guide):
from giskard.rag import KnowledgeBase, generate_testset, evaluate
kb = KnowledgeBase.from_pandas(df, columns=["text"])
testset = generate_testset(kb, num_questions=60, agent_description="...")
report = evaluate(my_rag_answer_fn, testset=testset, knowledge_base=kb)
report.to_html("giskard_rag_report.html")
```

---
## 10 · Outputs & reports — what Giskard produces

| Output | How to get it | Contents |
|--------|---------------|----------|
| **Scan report (HTML)** | `scan_results.to_html("scan.html")` | Interactive vulnerability dashboard grouped by category, with offending examples |
| **Scan issues (objects)** | `scan_results.issues` | Programmatic list for gating/automation |
| **Test suite** | `scan_results.generate_test_suite()` | Re-runnable pass/fail tests (regression) |
| **RAG test set** | `generate_testset(...).save("...jsonl")` | Synthetic Q&A grounded in your KB |
| **RAG report (HTML)** | `evaluate(...).to_html("rag.html")` | Correctness + per-component scores |

Everything is a local file or a Python object — no Hub, no upload.

---
## 11 · Iterating, customizing, going deeper

- **Sharpen the model `description`.** Like promptfoo's `purpose`, it's the biggest lever on scan
  quality — vague description, generic probes.
- **Scope scans with `only=[...]`** to iterate fast on one capability, then run the full suite.
- **Mind endpoint load.** The scan fires many requests at your FastAPI endpoint; ensure it can take
  the traffic (or add rate-limit handling / a small seed set) and that `TIMEOUT` is generous enough.
- **Swap the Gemini model** in `set_llm_model` to trade cost vs quality (e.g. a flash model while
  iterating, a pro model for the final scan). A stronger judge means fewer mis-judged issues.
- **Wire the test suite into CI.** `giskard` integrates with **pytest**; run
  `test_suite.run().passed` as a gate, and fail the build on new vulnerabilities. Save/load suites
  so they version alongside your app.
- **Verify findings.** LLM-assisted detection isn't perfect; spot-check flagged examples in the
  report rather than trusting rates blindly.

---
## Appendix A · Three-tool component map

| Concept | Giskard (OSS) | promptfoo | garak |
|---------|---------------|-----------|-------|
| Target model | `giskard.Model` | target / provider | generator |
| Attack generation | `giskard.scan` detectors | plugins + strategies | probes + buffs |
| Pass/fail judgment | LLM-assisted detectors | grader / assertion | detectors |
| Vulnerability report | ScanReport HTML | red-team report | `.report.jsonl` / summary |
| Regression tests | **test suite** | (CI via eval) | (CI via re-run) |
| RAG evaluation | **RAGET** | context-* assertions | (n/a) |
| "Bad" outcome | an **issue** | a **failed** probe | a **hit** |

Giskard's distinctive additions are the **auto-generated test suite** (regression gate) and
**RAGET** (component-attributed RAG scoring).

## Appendix B · Cheat-sheet & troubleshooting

```python
# Setup — evaluator on Gemini
pip install "giskard[llm]>2,<3"
import giskard
giskard.llm.set_llm_model("gemini/gemini-1.5-pro")
giskard.llm.set_embedding_model("gemini/text-embedding-004")

# Wrap a FastAPI endpoint + scan
import requests, pandas as pd
def model_predict(df):
    return [requests.post(URL, json={"question": q}, timeout=30).json()["answer"]
            for q in df["question"]]
m = giskard.Model(model_predict, model_type="text_generation",
                  name=..., description=..., feature_names=["question"])
report = giskard.scan(m)                       # or giskard.scan(m, dataset, only=["jailbreak"])
report.to_html("scan.html")
report.issues; report.has_vulnerabilities

# Regression suite
suite = report.generate_test_suite()
suite.run().passed

# RAGET (overview — Section 9): KnowledgeBase -> generate_testset -> evaluate -> report.to_html()
```

**Troubleshooting**
- *Scan errors about the LLM client* → `set_llm_model` / `set_embedding_model` not set, or
  `GEMINI_API_KEY` missing. Configure Section 3 before scanning.
- *Endpoint errors during scan* → the scan sends many requests; check the FastAPI URL, the JSON
  request/response keys in `call_endpoint`, timeouts, and that the service can handle the load.
- *Scan is slow / costly* → use `only=[...]` and a small seed dataset; scanning is LLM-assisted.
- *API differs from this notebook* → you may be on a different major version; this tutorial targets
  **v2** (`>2,<3`). Check the repo for v3 changes.

---
*Run cells top-to-bottom, replace every `REPLACE ME` / placeholder (model function, KB, answer_fn),
and keep artifacts in one working directory.*